# XGBoost training

Build a unified training frame with `build_training_frame()`: static IEEE columns from `train_features.parquet`, operational columns from PostgreSQL, and rolling behavioral features computed on the fly.

In [1]:
from fraud_scoring_engine.db.engine import create_db_engine
from fraud_scoring_engine.training import build_training_frame, time_split
from sqlalchemy.orm import Session

In [2]:
TARGET = "is_fraud"
LIMIT = 10_000

In [3]:
engine = create_db_engine()
with Session(engine) as session:
    df = build_training_frame(session, limit=LIMIT)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

Rows: 10,000
Columns: 438


In [4]:
display(df.shape)
display(df[TARGET].value_counts())
display(df[["transaction_id", "transaction_amt", "card1", "product_cd", "velocity_1h", TARGET]].head())

(10000, 438)

is_fraud
0    9735
1     265
Name: count, dtype: int64

,transaction_id,transaction_amt,card1,product_cd,velocity_1h,is_fraud
0,2987000,68.50,13926.0,W,0,0
1,2987001,29.00,2755.0,W,0,0
2,2987002,59.00,4663.0,W,0,0
3,2987003,50.00,18132.0,W,0,0
4,2987004,50.00,4497.0,H,0,0


## Train / validation / test split

Use `time_split()` to cut the frame into contiguous time blocks so later transactions cannot leak into training. Fit preprocessing on train only.

In [5]:
split = time_split(df)

for name, frame in [("train", split.train), ("val", split.val), ("test", split.test)]:
    fraud_rate = frame[TARGET].mean()
    print(
        f"{name:5s}  rows={len(frame):5,}  "
        f"fraud={int(frame[TARGET].sum()):4d}  "
        f"rate={fraud_rate:.4f}  "
        f"dt=[{frame['transaction_dt'].min()}, {frame['transaction_dt'].max()}]"
    )

train  rows=7,000  fraud= 172  rate=0.0246  dt=[86400, 233540]
val    rows=1,500  fraud=  44  rate=0.0293  dt=[233541, 253963]
test   rows=1,500  fraud=  49  rate=0.0327  dt=[253968, 313121]


## Feature preprocessing

Use `FraudFeaturePreprocessor` to drop metadata columns, keep numeric nulls for XGBoost, and encode categoricals with a stable `__MISSING__` level. Fit on train only so validation and test vocabularies cannot leak.

In [6]:
from fraud_scoring_engine.preprocessing import FraudFeaturePreprocessor

preprocessor = FraudFeaturePreprocessor()
X_train = preprocessor.fit_transform(split.train)
X_val = preprocessor.transform(split.val)
X_test = preprocessor.transform(split.test)

y_train = split.train[TARGET]
y_val = split.val[TARGET]
y_test = split.test[TARGET]

print(X_train.shape, X_val.shape, X_test.shape)
print("Number of categorical features:", X_train.select_dtypes("category").shape[1])
print("Number of float features:", X_train.select_dtypes("float").shape[1])
print("Number of int features:", X_train.select_dtypes("int").shape[1])

(7000, 435) (1500, 435) (1500, 435)
Number of categorical features: 32
Number of float features: 402
Number of int features: 1
